[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/32_nlp/02_bow_tfidf_classification.ipynb)

# Bài 02 — Bag-of-Words, TF-IDF và phân loại văn bản

**🎯 Mục tiêu**
- Biểu diễn văn bản thành vector: **Bag-of-Words**, ma trận thưa; cài **TF-IDF** from scratch và khớp `TfidfVectorizer`.
- Phân loại chủ đề tin tức: Naive Bayes, Logistic Regression, Linear SVM; unigram vs n-gram vs ký tự.
- **Diễn giải** mô hình tuyến tính (từ quan trọng mỗi lớp), **phân tích lỗi**.
- Giới hạn của BoW: **phủ định** ("không tốt" vs "không tệ") và vì sao bigram giúp.
- Đường cong học theo số nhãn → động lực cho embedding (bài 03).

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/32_nlp

In [ ]:
import re
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.feature_extraction.text import CountVectorizer, HashingVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, f1_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

from ml_course.datasets import NEWS_TOPICS, make_news_vi
from ml_course.visualization import set_style

set_style()
news = make_news_vi(4000, seed=0)
train, test = train_test_split(news, test_size=0.3, random_state=0, stratify=news.topic)
print(f"train {len(train)} | test {len(test)} | các chủ đề: {NEWS_TOPICS}")

## 1. Bag-of-Words

Bỏ qua thứ tự, chỉ đếm số lần mỗi từ xuất hiện: văn bản → vector dài bằng kích thước từ vựng. Hầu hết phần tử bằng 0 → lưu dạng **ma trận thưa** (chỉ lưu vị trí khác 0).

In [ ]:
docs = ["giá điện thoại tăng mạnh", "giá vàng giảm mạnh", "điện thoại mới ra mắt"]
vocab = sorted({w for d in docs for w in d.split()})
bow = np.array([[d.split().count(w) for w in vocab] for d in docs])
print(pd.DataFrame(bow, columns=vocab, index=[f"văn bản {i}" for i in range(3)]))

cv = CountVectorizer().fit(train.text)
X_counts = cv.transform(train.text)
density = X_counts.nnz / np.prod(X_counts.shape)
print(f"\nngữ liệu train: ma trận {X_counts.shape}, chỉ {density:.1%} phần tử khác 0 | "
      f"thưa: {X_counts.data.nbytes + X_counts.indices.nbytes + X_counts.indptr.nbytes:,} byte vs dày: {np.prod(X_counts.shape) * 8:,} byte")

## 2. TF-IDF

Từ xuất hiện ở **mọi** văn bản ("theo", "năm") ít giúp phân biệt. **TF-IDF** nhân tần suất trong văn bản (TF) với độ hiếm trên toàn ngữ liệu (IDF):

$$\text{tfidf}(t, d) = \text{tf}(t, d)\cdot\text{idf}(t), \qquad \text{idf}(t) = \ln\frac{1 + N}{1 + \text{df}(t)} + 1$$

($N$ số văn bản, $\text{df}(t)$ số văn bản chứa $t$; dạng "làm mịn" này là mặc định của scikit-learn), rồi chuẩn hoá mỗi vector về độ dài 1 (L2) để văn bản dài/ngắn so được với nhau.

In [ ]:
def tfidf_from_scratch(texts, vocabulary):
    counts = CountVectorizer(vocabulary=vocabulary).transform(texts).astype(float)   # tf: số lần xuất hiện
    N = counts.shape[0]
    df = np.asarray((counts > 0).sum(axis=0)).ravel()
    idf = np.log((1 + N) / (1 + df)) + 1
    X = counts.multiply(idf).tocsr()
    norms = np.sqrt(np.asarray(X.multiply(X).sum(axis=1)).ravel())
    return sparse.diags(1 / norms) @ X, idf


X_mine, idf = tfidf_from_scratch(train.text, cv.vocabulary_)
tv = TfidfVectorizer(vocabulary=cv.vocabulary_).fit(train.text)
print(f"sai lệch so với TfidfVectorizer: {abs(X_mine - tv.transform(train.text)).max():.1e}")
order = np.argsort(idf)
words = cv.get_feature_names_out()
print("idf thấp nhất (phổ biến):", ", ".join(f"{words[i]} {idf[i]:.2f}" for i in order[:5]))
print("idf cao nhất (hiếm)     :", ", ".join(f"{words[i]} {idf[i]:.2f}" for i in order[-5:]))

## 3. Phân loại chủ đề: mô hình × biểu diễn

- **Multinomial Naive Bayes** (chương về Naive Bayes): nhanh, mạnh khi ít dữ liệu.
- **Logistic Regression** và **Linear SVM**: mô hình tuyến tính trên vector TF-IDF — baseline rất khó vượt cho phân loại văn bản.
- Biểu diễn: unigram (âm tiết), unigram + bigram (bigram âm tiết bắt được từ ghép "điện thoại", "lãi suất"), n-gram ký tự (bền với lỗi chính tả).

In [ ]:
vectorizers = {
    "unigram": TfidfVectorizer(),
    "unigram + bigram": TfidfVectorizer(ngram_range=(1, 2)),
    "char 2–5": TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5)),
}
models = {"Naive Bayes": lambda: MultinomialNB(alpha=0.1), "LogReg": lambda: LogisticRegression(max_iter=3000, C=10),
          "Linear SVM": lambda: LinearSVC(C=1.0)}
rows = []
for vname, vec in vectorizers.items():
    Xtr, Xte = vec.fit_transform(train.text), vec.transform(test.text)
    for mname, make in models.items():
        t0 = time.perf_counter()
        pred = make().fit(Xtr, train.topic).predict(Xte)
        rows.append((vname, mname, Xtr.shape[1], (pred == test.topic).mean(), f1_score(test.topic, pred, average="macro"),
                     time.perf_counter() - t0))
results = pd.DataFrame(rows, columns=["biểu diễn", "mô hình", "số đặc trưng", "accuracy", "macro-F1", "thời gian (s)"])
print(results.round(3).to_string(index=False))

- Mọi tổ hợp đều quanh 0.93–0.97: với văn bản, **biểu diễn + mô hình tuyến tính** đã rất mạnh. Thêm bigram cải thiện thêm khoảng 1–3 điểm (bắt được từ ghép và cụm cố định).
- Trần độ chính xác bị giới hạn bởi dữ liệu: khoảng 3% nhãn bị gán sai và nhiều tin có câu lạc đề — không mô hình nào "đúng" 100% ở đây.
- Huấn luyện tính bằng giây trên hàng nghìn văn bản — luôn bắt đầu từ baseline này trước khi nghĩ tới mạng nơ-ron.

## 4. Diễn giải: từ nào quyết định?

Với mô hình tuyến tính, hệ số của mỗi đặc trưng cho mỗi lớp cho biết từ nào đẩy văn bản về lớp đó. (Để ý chữ "không" trong lớp du lịch: đến từ "hãng hàng **không**" — tách theo âm tiết làm từ ghép vỡ thành các mảnh mang nghĩa khác, bài 01.)

In [ ]:
vec = TfidfVectorizer(ngram_range=(1, 2))
Xtr, Xte = vec.fit_transform(train.text), vec.transform(test.text)
clf = LogisticRegression(max_iter=3000, C=10).fit(Xtr, train.topic)
features = vec.get_feature_names_out()
for k, topic in enumerate(clf.classes_):
    top = np.argsort(clf.coef_[k])[-6:][::-1]
    print(f"{topic:9s}: {', '.join(features[top])}")

In [ ]:
pred = clf.predict(Xte)
fig, ax = plt.subplots(figsize=(6.5, 5.5))
ConfusionMatrixDisplay.from_predictions(test.topic, pred, ax=ax, xticks_rotation=45, colorbar=False)
plt.tight_layout()
plt.show()

wrong = test[pred != test.topic].assign(dự_đoán=pred[pred != test.topic])
print(f"{len(wrong)} lỗi / {len(test)}. Vài ví dụ:")
for t in wrong.head(6).itertuples():
    print(f"  nhãn {t.topic:9s} | dự đoán {t.dự_đoán:9s} | {t.text}")

Đọc các lỗi: phần lớn rơi vào hai loại — **nhãn sai** (văn bản rõ ràng thuộc chủ đề mô hình dự đoán) và **câu lạc đề** ("Trong khi đó, ...") có trọng lượng lớn hơn nội dung chính. Phân tích lỗi cho biết nên đầu tư vào đâu: làm sạch nhãn, hay xử lý cấu trúc văn bản (vd giảm trọng số mệnh đề sau "Trong khi đó") — chứ không phải đổi sang mô hình phức tạp hơn.

## 5. Giới hạn của Bag-of-Words: phủ định

BoW mất thứ tự. Xét đánh giá ngắn trong đó "không" **đảo nghĩa theo cả hai chiều**: "không tốt" là chê, "không tệ" là khen.

In [ ]:
POS_WORDS = ["tốt", "đẹp", "bền", "nhanh", "chắc chắn", "đáng tiền"]
NEG_WORDS = ["tệ", "xấu", "lỗi", "chậm", "dởm", "phí tiền"]


def make_negation_reviews(n, seed):
    r = np.random.default_rng(seed)
    texts, labels = [], []
    for _ in range(n):
        label = int(r.random() < 0.5)
        phrases = []
        for _ in range(r.integers(1, 3)):
            negate = r.random() < 0.4
            pool = (NEG_WORDS if negate else POS_WORDS) if label == 1 else (POS_WORDS if negate else NEG_WORDS)
            word = pool[r.integers(len(pool))]
            phrases.append(("không " if negate else r.choice(["rất ", "khá ", ""])) + word)
        texts.append("sản phẩm " + " và ".join(phrases))
        labels.append(label)
    return pd.DataFrame({"text": texts, "label": labels})


neg_train, neg_test = make_negation_reviews(3000, 0), make_negation_reviews(1000, 1)
print(neg_train.head(4).to_string(index=False))
has_negation = neg_test.text.str.contains("không ").values
for name, ngram in [("unigram", (1, 1)), ("unigram + bigram", (1, 2))]:
    v = CountVectorizer(ngram_range=ngram, binary=True).fit(neg_train.text)
    m = LogisticRegression(max_iter=3000).fit(v.transform(neg_train.text), neg_train.label)
    p = m.predict(v.transform(neg_test.text))
    print(f"{name:17s}: accuracy = {(p == neg_test.label).mean():.3f} | câu có phủ định = {(p[has_negation] == neg_test.label[has_negation]).mean():.3f}"
          f" | không phủ định = {(p[~has_negation] == neg_test.label[~has_negation]).mean():.3f}")

Unigram chỉ thấy "không" và "tốt" riêng rẽ — một mô hình tuyến tính trên các đặc trưng đó **không thể** biểu diễn "tốt là khen, nhưng không + tốt là chê" (giống bài toán XOR). Bigram thêm đặc trưng "không tốt" → giải quyết. Đây chỉ là trường hợp đơn giản nhất; phủ định xa ("không ai nói sản phẩm này tốt"), mỉa mai, so sánh cần mô hình hiểu cấu trúc câu — RNN/Transformer (chương 30–31).

## 6. Bao nhiêu nhãn là đủ?

Gán nhãn tốn kém. Đường cong học theo số văn bản có nhãn (5 lần chọn mẫu ngẫu nhiên mỗi mức):

In [ ]:
sizes = [30, 100, 300, 1000, len(train)]
curve = {}
for n in sizes:
    scores = []
    for seed in range(5 if n < len(train) else 1):
        sub = train.sample(n, random_state=seed) if n < len(train) else train
        v = TfidfVectorizer(ngram_range=(1, 2)).fit(sub.text)
        m = LogisticRegression(max_iter=3000, C=10).fit(v.transform(sub.text), sub.topic)
        scores.append((m.predict(v.transform(test.text)) == test.topic).mean())
    curve[n] = (np.mean(scores), np.std(scores))
    print(f"{n:5d} nhãn: accuracy = {curve[n][0]:.3f} ± {curve[n][1]:.3f}")

fig, ax = plt.subplots(figsize=(6, 3.6))
ax.errorbar(sizes, [curve[n][0] for n in sizes], yerr=[curve[n][1] for n in sizes], marker="o", capsize=3)
ax.set_xscale("log")
ax.set_xlabel("số văn bản có nhãn")
ax.set_ylabel("accuracy test")
plt.tight_layout()
plt.show()

Với vài chục nhãn, TF-IDF chỉ "biết" những từ có trong vài văn bản đó — văn bản test dùng **từ đồng nghĩa** ("smartphone" thay cho "điện thoại thông minh") không khớp được đặc trưng nào. Bài 03: học biểu diễn từ từ **văn bản không nhãn** (rẻ, nhiều) để các từ cùng nghĩa có vector gần nhau.

## 7. Ghi chú triển khai

- **Fit vectorizer trên tập train** (như mọi bước tiền xử lý — chương 4/5); từ chỉ có ở test bị bỏ qua.
- Từ vựng lớn/liên tục thay đổi (log, mạng xã hội): `HashingVectorizer` băm từ vào số chiều cố định — không cần lưu từ vựng, chấp nhận va chạm hiếm.
- Đặt vectorizer + mô hình trong một `Pipeline` để tìm siêu tham số bằng cross-validation (`ngram_range`, `min_df`, `C`).

In [ ]:
hv = HashingVectorizer(n_features=2**12, ngram_range=(1, 2), alternate_sign=False, norm="l2")
m = LogisticRegression(max_iter=3000, C=10).fit(hv.transform(train.text), train.topic)
print(f"HashingVectorizer 4 096 chiều: accuracy = {(m.predict(hv.transform(test.text)) == test.topic).mean():.3f} (không cần lưu từ vựng)")

## 🧠 Kiểm tra nhanh

1. Viết công thức TF-IDF. Vì sao chuẩn hoá L2?
2. Vì sao mô hình tuyến tính trên TF-IDF là baseline mạnh cho phân loại văn bản?
3. Đọc hệ số của Logistic Regression để diễn giải thế nào?
4. Vì sao unigram BoW không xử lý được "không tốt" vs "không tệ"? Bigram giúp ra sao? Giới hạn còn lại?
5. Vì sao accuracy với 30 nhãn thấp? Hướng cải thiện?

## 📌 Tóm tắt

- BoW/TF-IDF: vector thưa theo từ vựng; IDF giảm trọng số từ phổ biến; chuẩn hoá L2.
- NB/LogReg/SVM tuyến tính + n-gram: nhanh, mạnh, dễ diễn giải — luôn là baseline đầu tiên.
- Phân tích lỗi trước khi đổi mô hình: nhãn nhiễu, câu lạc đề.
- BoW mất thứ tự: phủ định cần n-gram hoặc mô hình chuỗi; ít nhãn → cần biểu diễn học từ dữ liệu không nhãn.

## 📚 Tài liệu tham khảo

- Jurafsky & Martin — *Speech and Language Processing*, chương 4 (Naive Bayes, Text Classification), chương 6 (Vector Semantics)
- Manning, Raghavan & Schütze — *Introduction to Information Retrieval*, chương 6 (tf-idf)
- Wang & Manning (2012) — *Baselines and Bigrams: Simple, Good Sentiment and Topic Classification*
- [scikit-learn — Working With Text Data](https://scikit-learn.org/stable/tutorial/text_analytics/working_with_text_data.html)